# Extração e Análise de Dados do Perfil William

In [20]:
pip install requests pandas openpyxl matplotlib

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. Extração de Dados da Fonte 1

In [21]:
import datetime
import getpass
import pandas as pd
import requests

print("Configurações iniciais:")
STEAM_ID = "76561198092671592"

ano_atual = datetime.datetime.now().year
limite_3_anos = ano_atual - 3

STEAM_API_KEY = getpass.getpass(
    "Cole sua Chave Steam aqui (ela ficará invisível) e aperte Enter: "
)

if len(STEAM_API_KEY.strip()) < 32:
    print("\n[AVISO] Uma chave de API da Steam padrão possui exatamente 32 caracteres.")
    print("Certifique-se de copiá-la em: https://steamcommunity.com")

url = "https://api.steampowered.com/IPlayerService/GetOwnedGames/v1/"
params = {
    "key": STEAM_API_KEY.strip(),
    "steamid": STEAM_ID,
    "include_appinfo": "true",
    "include_played_free_games": "true",
    "format": "json",
}

print("\nBuscando dados no servidor da API da Steam...")
try:
    response = requests.get(url, params=params, timeout=10)
    content_type = response.headers.get("Content-Type", "")
    if "application/json" not in content_type:
        print("\n[ERRO CRÍTICO] A Steam barrou a requisição. Verifique sua chave.")
        data = None
    else:
        data = response.json()
except Exception as e:
    print(f"\n[ERRO DE CONEXÃO] Falha ao acessar a infraestrutura: {e}")
    data = None

if data and "response" in data and "games" in data["response"]:
    games_list = data["response"]["games"]
    processed_games = []

    for game in games_list:
        horas_totais = round(game.get("playtime_forever", 0) / 60, 1)
        horas_2_semanas = round(game.get("playtime_2weeks", 0) / 60, 1)

        last_played_timestamp = game.get("rtime_last_played", 0)
        if last_played_timestamp > 0:
            dt_objeto = datetime.datetime.fromtimestamp(last_played_timestamp)
            data_ultimo_login = dt_objeto.strftime("%Y-%m-%d %H:%M:%S")
            ano_ultimo_login = dt_objeto.year
            status_3_anos = "Ativo (Últimos 3 Anos)" if ano_ultimo_login >= limite_3_anos else "Inativo (Mais de 3 Anos)"
        else:
            data_ultimo_login = "Nunca Jogado"
            status_3_anos = "Nunca Jogado"

        processed_games.append({
            "ID_Jogo": game.get("appid"),
            "Nome_Jogo": game.get("name", "Nome Desconhecido"),
            "Horas_Jogadas_Totais": horas_totais,
            "Horas_Jogadas_Ultimas_2Semanas": horas_2_semanas,
            "Data_Ultimo_Login": data_ultimo_login,
            "Status_Ciclo_3_Anos": status_3_anos,
        })

    dfWilliam = pd.DataFrame(processed_games)
    dfWilliam = dfWilliam.sort_values(by="Horas_Jogadas_Totais", ascending=False).reset_index(drop=True)

    print("\n✅ Conexão bem-sucedida! Dados coletados (Fonte 1 - Steam API) em 'dfWilliam'.")
    print(dfWilliam[["Nome_Jogo", "Horas_Jogadas_Totais", "Data_Ultimo_Login", "Status_Ciclo_3_Anos"]].head(10))

    dfWilliam.to_csv("dados_steam_analise.csv", index=False, encoding="utf-8-sig")
    print("\n[Sucesso] Tabela estruturada gravada em 'dados_steam_analise.csv'!")

elif data is not None:
    print("\n[AVISO] Autenticação feita, mas nenhum jogo foi retornado.")
    print("CAUSA PROVÁVEL: privacidade do perfil Steam está como privada.")


Configurações iniciais:

Buscando dados no servidor da API da Steam...

✅ Conexão bem-sucedida! Dados coletados (Fonte 1 - Steam API) em 'dfWilliam'.
            Nome_Jogo  Horas_Jogadas_Totais    Data_Ultimo_Login  \
0    Counter-Strike 2                1658.6  2026-07-05 19:58:24   
1              Dota 2                 764.0  2024-11-07 22:55:29   
2        Apex Legends                 223.5  2025-10-18 21:50:07   
3       Left 4 Dead 2                 194.5  2026-06-04 21:45:42   
4            Lost Ark                 170.4  2024-01-17 22:07:44   
5       Call of Duty®                 159.9  2025-09-21 21:26:35   
6   Shakes and Fidget                 158.4  2022-05-29 13:15:05   
7         ARC Raiders                 122.5  2026-04-24 23:42:06   
8     Path of Exile 2                  87.5  2026-05-31 22:27:04   
9  DC Universe Online                  75.7  2026-04-12 21:35:05   

        Status_Ciclo_3_Anos  
0    Ativo (Últimos 3 Anos)  
1    Ativo (Últimos 3 Anos)  
2    Ativo 

## 2. Limpeza da Fonte 1 

In [22]:
print("Valores ausentes em dfWilliam ANTES da limpeza:")
print(dfWilliam.isnull().sum())

antes = len(dfWilliam)
dfWilliam = dfWilliam.drop_duplicates(subset="ID_Jogo").reset_index(drop=True)
print(f"\nDuplicatas removidas em dfWilliam: {antes - len(dfWilliam)}")

dfWilliam["Horas_Jogadas_Totais"] = pd.to_numeric(dfWilliam["Horas_Jogadas_Totais"], errors="coerce").fillna(0)
dfWilliam["Horas_Jogadas_Ultimas_2Semanas"] = pd.to_numeric(dfWilliam["Horas_Jogadas_Ultimas_2Semanas"], errors="coerce").fillna(0)

dfWilliam["Nome_Jogo"] = dfWilliam["Nome_Jogo"].astype(str).str.strip()

print("\nValores ausentes em dfWilliam DEPOIS da limpeza:")
print(dfWilliam.isnull().sum())
dfWilliam.head()


Valores ausentes em dfWilliam ANTES da limpeza:
ID_Jogo                           0
Nome_Jogo                         0
Horas_Jogadas_Totais              0
Horas_Jogadas_Ultimas_2Semanas    0
Data_Ultimo_Login                 0
Status_Ciclo_3_Anos               0
dtype: int64

Duplicatas removidas em dfWilliam: 0

Valores ausentes em dfWilliam DEPOIS da limpeza:
ID_Jogo                           0
Nome_Jogo                         0
Horas_Jogadas_Totais              0
Horas_Jogadas_Ultimas_2Semanas    0
Data_Ultimo_Login                 0
Status_Ciclo_3_Anos               0
dtype: int64


,ID_Jogo,Nome_Jogo,Horas_Jogadas_Totais,Horas_Jogadas_Ultimas_2Semanas,Data_Ultimo_Login,Status_Ciclo_3_Anos
0,730,Counter-Strike 2,1658.6,0.2,2026-07-05 19:58:24,Ativo (Últimos 3 Anos)
1,570,Dota 2,764.0,0.0,2024-11-07 22:55:29,Ativo (Últimos 3 Anos)
2,1172470,Apex Legends,223.5,0.0,2025-10-18 21:50:07,Ativo (Últimos 3 Anos)
3,550,Left 4 Dead 2,194.5,0.0,2026-06-04 21:45:42,Ativo (Últimos 3 Anos)
4,1599340,Lost Ark,170.4,0.0,2024-01-17 22:07:44,Ativo (Últimos 3 Anos)


## 3. Extração de Dados Fonte 2: SteamSpy API


In [23]:
import time

def fetch_steamspy(appid):
    """Consulta a API pública da SteamSpy para um appid específico."""
    url = "https://steamspy.com/api.php"
    params = {"request": "appdetails", "appid": appid}
    try:
        r = requests.get(url, params=params, timeout=10)
        if r.status_code == 200:
            return r.json()
    except Exception as e:
        print(f"[AVISO] Falha ao consultar appid {appid}: {e}")
    return None

print("Consultando SteamSpy para cada jogo de dfWilliam (~1 seg por jogo)...")
steamspy_rows = []

for appid in dfWilliam["ID_Jogo"]:
    info = fetch_steamspy(appid)
    if info:
        steamspy_rows.append({
            "ID_Jogo": appid,
            "Genero": info.get("genre"),
            "Preco_Centavos": info.get("price"),
            "Avaliacoes_Positivas": info.get("positive"),
            "Avaliacoes_Negativas": info.get("negative"),
            "Faixa_Donos_Estimada": info.get("owners"),
        })
    else:
        # Sem resposta da API: mantemos como NaN (None), e NÃO como 0,
        # para não confundir "sem dado" com "jogo gratuito" mais adiante.
        steamspy_rows.append({
            "ID_Jogo": appid, "Genero": None, "Preco_Centavos": None,
            "Avaliacoes_Positivas": None, "Avaliacoes_Negativas": None,
            "Faixa_Donos_Estimada": None,
        })
    time.sleep(1)

df_steamspy = pd.DataFrame(steamspy_rows)
print(f"\n[Sucesso] {len(df_steamspy)} registros consultados na SteamSpy (Fonte 2).")
df_steamspy.head()


Consultando SteamSpy para cada jogo de dfWilliam (~1 seg por jogo)...

[Sucesso] 52 registros consultados na SteamSpy (Fonte 2).


,ID_Jogo,Genero,Preco_Centavos,Avaliacoes_Positivas,Avaliacoes_Negativas,Faixa_Donos_Estimada
0,730,"Action, Free To Play",0,7642084,1173003,"100,000,000 .. 200,000,000"
1,570,"Action, Strategy, Free To Play",0,2037143,461826,"100,000,000 .. 200,000,000"
2,1172470,"Action, Adventure, Free To Play",0,668053,326926,"100,000,000 .. 200,000,000"
3,550,Action,999,940221,23762,"50,000,000 .. 100,000,000"
4,1599340,"Action, Adventure, Massively Multiplayer, RPG,...",0,143481,58540,"50,000,000 .. 100,000,000"


## 4. Limpeza da Fonte 2 

In [24]:
print("Valores ausentes em df_steamspy ANTES da limpeza:")
print(df_steamspy.isnull().sum())

# 1) Remoção de duplicatas (por ID_Jogo)
antes = len(df_steamspy)
df_steamspy = df_steamspy.drop_duplicates(subset="ID_Jogo").reset_index(drop=True)
print(f"\nDuplicatas removidas em df_steamspy: {antes - len(df_steamspy)}")

# 2) Conversão de tipos de dados (mantendo NaN onde não há dado real)
df_steamspy["Preco_Centavos"] = pd.to_numeric(df_steamspy["Preco_Centavos"], errors="coerce")
df_steamspy["Avaliacoes_Positivas"] = pd.to_numeric(df_steamspy["Avaliacoes_Positivas"], errors="coerce")
df_steamspy["Avaliacoes_Negativas"] = pd.to_numeric(df_steamspy["Avaliacoes_Negativas"], errors="coerce")

# 3) Padronização de texto (só aplicada onde existe valor; genero ausente permanece NaN por enquanto)
df_steamspy["Genero"] = df_steamspy["Genero"].astype(str).str.strip().str.title()
df_steamspy.loc[df_steamspy["Genero"].isin(["None", "Nan"]), "Genero"] = None

print("\nValores ausentes em df_steamspy DEPOIS da limpeza:")
print(df_steamspy.isnull().sum())
df_steamspy.head()


Valores ausentes em df_steamspy ANTES da limpeza:
ID_Jogo                 0
Genero                  0
Preco_Centavos          6
Avaliacoes_Positivas    0
Avaliacoes_Negativas    0
Faixa_Donos_Estimada    0
dtype: int64

Duplicatas removidas em df_steamspy: 0

Valores ausentes em df_steamspy DEPOIS da limpeza:
ID_Jogo                 0
Genero                  0
Preco_Centavos          6
Avaliacoes_Positivas    0
Avaliacoes_Negativas    0
Faixa_Donos_Estimada    0
dtype: int64


,ID_Jogo,Genero,Preco_Centavos,Avaliacoes_Positivas,Avaliacoes_Negativas,Faixa_Donos_Estimada
0,730,"Action, Free To Play",0.0,7642084,1173003,"100,000,000 .. 200,000,000"
1,570,"Action, Strategy, Free To Play",0.0,2037143,461826,"100,000,000 .. 200,000,000"
2,1172470,"Action, Adventure, Free To Play",0.0,668053,326926,"100,000,000 .. 200,000,000"
3,550,Action,999.0,940221,23762,"50,000,000 .. 100,000,000"
4,1599340,"Action, Adventure, Massively Multiplayer, Rpg,...",0.0,143481,58540,"50,000,000 .. 100,000,000"


## 5. Merge das Duas Fontes

In [25]:
df = dfWilliam.merge(df_steamspy, on="ID_Jogo", how="left")

# Genero: aqui sim, um jogo realmente sem categoria de gênero informada vira "Não Informado"
df["Genero"] = df["Genero"].fillna("Não Informado")

print(f"[Sucesso] Base final unificada: {len(df)} jogos.")
print("\nValores ausentes após o merge (preço/avaliações ausentes = SteamSpy não teve dado, mantido como NaN):")
print(df[["Preco_Centavos", "Avaliacoes_Positivas", "Avaliacoes_Negativas"]].isnull().sum())
df.head()


[Sucesso] Base final unificada: 52 jogos.

Valores ausentes após o merge (preço/avaliações ausentes = SteamSpy não teve dado, mantido como NaN):
Preco_Centavos          6
Avaliacoes_Positivas    0
Avaliacoes_Negativas    0
dtype: int64


,ID_Jogo,Nome_Jogo,Horas_Jogadas_Totais,Horas_Jogadas_Ultimas_2Semanas,Data_Ultimo_Login,Status_Ciclo_3_Anos,Genero,Preco_Centavos,Avaliacoes_Positivas,Avaliacoes_Negativas,Faixa_Donos_Estimada
0,730,Counter-Strike 2,1658.6,0.2,2026-07-05 19:58:24,Ativo (Últimos 3 Anos),"Action, Free To Play",0.0,7642084,1173003,"100,000,000 .. 200,000,000"
1,570,Dota 2,764.0,0.0,2024-11-07 22:55:29,Ativo (Últimos 3 Anos),"Action, Strategy, Free To Play",0.0,2037143,461826,"100,000,000 .. 200,000,000"
2,1172470,Apex Legends,223.5,0.0,2025-10-18 21:50:07,Ativo (Últimos 3 Anos),"Action, Adventure, Free To Play",0.0,668053,326926,"100,000,000 .. 200,000,000"
3,550,Left 4 Dead 2,194.5,0.0,2026-06-04 21:45:42,Ativo (Últimos 3 Anos),Action,999.0,940221,23762,"50,000,000 .. 100,000,000"
4,1599340,Lost Ark,170.4,0.0,2024-01-17 22:07:44,Ativo (Últimos 3 Anos),"Action, Adventure, Massively Multiplayer, Rpg,...",0.0,143481,58540,"50,000,000 .. 100,000,000"


## 6. Transformação dos Dados

1. `Preco_Dolar`
2. `Percentual_Positivo`
3. `Dias_Desde_Ultimo_Login`
4. `Faixa_Preco`
5. `Categoria_Consumo`
6. `Custo_Por_Hora`

In [26]:
from datetime import datetime

hoje = datetime.now()

# 1. Preço em dólares
df["Preco_Dolar"] = df["Preco_Centavos"] / 100

# 2. Percentual de avaliações positivas (só é calculado quando tem avaliações reais)
df["Total_Avaliacoes"] = df["Avaliacoes_Positivas"] + df["Avaliacoes_Negativas"]
df["Percentual_Positivo"] = df.apply(
    lambda row: round((row["Avaliacoes_Positivas"] / row["Total_Avaliacoes"]) * 100, 1)
    if pd.notna(row["Total_Avaliacoes"]) and row["Total_Avaliacoes"] > 0 else None,
    axis=1,
)

# 3. Dias desde o último login
def calcular_dias(data_str):
    if data_str == "Nunca Jogado":
        return None
    dt = datetime.strptime(data_str, "%Y-%m-%d %H:%M:%S")
    return (hoje - dt).days

df["Dias_Desde_Ultimo_Login"] = df["Data_Ultimo_Login"].apply(calcular_dias)

# 4. Faixa de preço (sem inventar "Gratuito" quando o preço é, na verdade, desconhecido)
def faixa_preco(preco):
    if pd.isna(preco):
        return "Dado Indisponível"
    elif preco == 0:
        return "Gratuito"
    elif preco < 10:
        return "Econômico (< $10)"
    elif preco < 30:
        return "Intermediário ($10 - $30)"
    else:
        return "Premium (> $30)"

df["Faixa_Preco"] = df["Preco_Dolar"].apply(faixa_preco)

# 5. Categoria de consumo por horas jogadas
def segmentar_jogo(horas):
    if horas >= 500:
        return "Hardcore (> 500h)"
    elif horas >= 100:
        return "Engajado Avançado (100h - 500h)"
    elif horas >= 10:
        return "Casual Intermediário (10h - 100h)"
    elif horas > 0:
        return "Experimental / Testado (< 10h)"
    else:
        return "Nunca Jogado (0h)"

df["Categoria_Consumo"] = df["Horas_Jogadas_Totais"].apply(segmentar_jogo)

# 6. Custo por hora jogada
df["Custo_Por_Hora"] = df.apply(
    lambda row: round(row["Preco_Dolar"] / row["Horas_Jogadas_Totais"], 2)
    if pd.notna(row["Preco_Dolar"]) and row["Horas_Jogadas_Totais"] > 0 else None,
    axis=1,
)

print("Novas colunas criadas:")
print(["Preco_Dolar", "Percentual_Positivo", "Dias_Desde_Ultimo_Login",
       "Faixa_Preco", "Categoria_Consumo", "Custo_Por_Hora"])

df.to_csv("dados_steam_completo.csv", index=False, encoding="utf-8-sig")
print("\n[Sucesso] Base final (2 fontes limpas + unidas + transformadas) salva em 'dados_steam_completo.csv'")
df.head()


Novas colunas criadas:
['Preco_Dolar', 'Percentual_Positivo', 'Dias_Desde_Ultimo_Login', 'Faixa_Preco', 'Categoria_Consumo', 'Custo_Por_Hora']

[Sucesso] Base final (2 fontes limpas + unidas + transformadas) salva em 'dados_steam_completo.csv'


,ID_Jogo,Nome_Jogo,Horas_Jogadas_Totais,Horas_Jogadas_Ultimas_2Semanas,Data_Ultimo_Login,Status_Ciclo_3_Anos,Genero,Preco_Centavos,Avaliacoes_Positivas,Avaliacoes_Negativas,Faixa_Donos_Estimada,Preco_Dolar,Total_Avaliacoes,Percentual_Positivo,Dias_Desde_Ultimo_Login,Faixa_Preco,Categoria_Consumo,Custo_Por_Hora
0,730,Counter-Strike 2,1658.6,0.2,2026-07-05 19:58:24,Ativo (Últimos 3 Anos),"Action, Free To Play",0.0,7642084,1173003,"100,000,000 .. 200,000,000",0.00,8815087,86.7,2.0,Gratuito,Hardcore (> 500h),0.00
1,570,Dota 2,764.0,0.0,2024-11-07 22:55:29,Ativo (Últimos 3 Anos),"Action, Strategy, Free To Play",0.0,2037143,461826,"100,000,000 .. 200,000,000",0.00,2498969,81.5,607.0,Gratuito,Hardcore (> 500h),0.00
2,1172470,Apex Legends,223.5,0.0,2025-10-18 21:50:07,Ativo (Últimos 3 Anos),"Action, Adventure, Free To Play",0.0,668053,326926,"100,000,000 .. 200,000,000",0.00,994979,67.1,262.0,Gratuito,Engajado Avançado (100h - 500h),0.00
3,550,Left 4 Dead 2,194.5,0.0,2026-06-04 21:45:42,Ativo (Últimos 3 Anos),Action,999.0,940221,23762,"50,000,000 .. 100,000,000",9.99,963983,97.5,33.0,Econômico (< $10),Engajado Avançado (100h - 500h),0.05
4,1599340,Lost Ark,170.4,0.0,2024-01-17 22:07:44,Ativo (Últimos 3 Anos),"Action, Adventure, Massively Multiplayer, Rpg,...",0.0,143481,58540,"50,000,000 .. 100,000,000",0.00,202021,71.0,902.0,Gratuito,Engajado Avançado (100h - 500h),0.00


## 7. Análise Exploratória (EDA) e Visualizações

In [27]:
import os
import matplotlib.pyplot as plt
import pandas as pd

csv_path = "dados_steam_completo.csv"

if not os.path.exists(csv_path):
    print(f"[ERRO] Arquivo '{csv_path}' não encontrado. Rode as etapas anteriores primeiro!")
else:
    df = pd.read_csv(csv_path)
    plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")

    print("=" * 60)
    print("   ANÁLISE ESTATÍSTICA DA BIBLIOTECA STEAM")
    print("=" * 60)

    # ---------------------------------------------------------
    # Análise 1: Concentração de Horas (Top 3 vs Resto)
    # ---------------------------------------------------------
    horas_totais_biblioteca = df["Horas_Jogadas_Totais"].sum()
    horas_top3 = df["Horas_Jogadas_Totais"].head(3).sum()
    horas_resto = horas_totais_biblioteca - horas_top3
    pct_top3 = round((horas_top3 / horas_totais_biblioteca) * 100, 1)

    print(f"\n1. Concentração do Tempo de Jogo:")
    print(f"   - Top 3 jogos consomem {horas_top3:.1f}h das {horas_totais_biblioteca:.1f}h totais ({pct_top3}%).")

    plt.figure(figsize=(6, 4))
    plt.pie([horas_top3, horas_resto], labels=["Top 3 Jogos", "Restante da Biblioteca"],
            autopct="%1.1f%%", colors=["#171a21", "#66c0f4"], startangle=90)
    plt.title("Concentração de Horas Jogadas (Top 3 vs Resto)")
    plt.tight_layout()
    plt.savefig("grafico_pergunta_1.png")
    plt.close()

    # ---------------------------------------------------------
    # Análise 2: Jogados vs Nunca Jogados
    # ---------------------------------------------------------
    total_jogos = len(df)
    jogados = len(df[df["Horas_Jogadas_Totais"] > 0])
    nunca_jogados = total_jogos - jogados
    pct_jogados = round((jogados / total_jogos) * 100, 1)

    print(f"\n2. Aproveitamento da Biblioteca:")
    print(f"   - Dos {total_jogos} jogos, {jogados} foram jogados efetivamente ({pct_jogados}%).")
    print(f"   - {nunca_jogados} jogos possuem 0 horas registradas.")

    # ---------------------------------------------------------
    # Análise 3: Ciclo Temporal de 3 Anos
    # ---------------------------------------------------------
    contagem_ciclo = df["Status_Ciclo_3_Anos"].value_counts()
    print(f"\n3. Retenção e Atividade (Corte Temporal):")
    for status, qtd in contagem_ciclo.items():
        print(f"   - {status}: {qtd} jogo(s)")

    plt.figure(figsize=(7, 4))
    contagem_ciclo.plot(kind="bar", color="#1b2838", edgecolor="gray")
    plt.title("Status de Atividade dos Jogos no Ciclo de 3 Anos")
    plt.ylabel("Quantidade de Jogos")
    plt.xticks(rotation=15)
    plt.tight_layout()
    plt.savefig("grafico_pergunta_3.png")
    plt.close()

    # ---------------------------------------------------------
    # Análise 4: Top 5 Histórico
    # ---------------------------------------------------------
    print(f"\n4. Linha do Tempo dos 5 Maiores Sucessos:")
    top5 = df.head(5)
    for idx, row in top5.iterrows():
        print(f"   - {row['Nome_Jogo']}: {row['Horas_Jogadas_Totais']}h | Último login: {row['Data_Ultimo_Login']}")

    # ---------------------------------------------------------
    # Análise 5: Jogados Pouco / Só Testados
    # ---------------------------------------------------------
    distribuicao_perfil = df["Categoria_Consumo"].value_counts()
    print(f"\n5. Jogados Pouco / Só Testados:")
    for cat, qtd in distribuicao_perfil.items():
        print(f"   - {cat}: {qtd} título(s)")

    plt.figure(figsize=(8, 4))
    distribuicao_perfil.sort_values().plot(kind="barh", color="#2a475e")
    plt.title("Distribuição da Biblioteca por Categoria de Consumo")
    plt.xlabel("Quantidade de Títulos")
    plt.tight_layout()
    plt.savefig("grafico_pergunta_5.png")
    plt.close()

    print("\n" + "=" * 60)
    print("[SUCESSO] Gráficos salvos: grafico_pergunta_1.png, _3.png e _5.png")
    print("=" * 60)


   ANÁLISE ESTATÍSTICA DA BIBLIOTECA STEAM

1. Concentração do Tempo de Jogo:
   - Top 3 jogos consomem 2646.1h das 3806.2h totais (69.5%).

2. Aproveitamento da Biblioteca:
   - Dos 52 jogos, 39 foram jogados efetivamente (75.0%).
   - 13 jogos possuem 0 horas registradas.

3. Retenção e Atividade (Corte Temporal):
   - Inativo (Mais de 3 Anos): 20 jogo(s)
   - Ativo (Últimos 3 Anos): 19 jogo(s)
   - Nunca Jogado: 13 jogo(s)

4. Linha do Tempo dos 5 Maiores Sucessos:
   - Counter-Strike 2: 1658.6h | Último login: 2026-07-05 19:58:24
   - Dota 2: 764.0h | Último login: 2024-11-07 22:55:29
   - Apex Legends: 223.5h | Último login: 2025-10-18 21:50:07
   - Left 4 Dead 2: 194.5h | Último login: 2026-06-04 21:45:42
   - Lost Ark: 170.4h | Último login: 2024-01-17 22:07:44

5. Jogados Pouco / Só Testados:
   - Experimental / Testado (< 10h): 23 título(s)
   - Nunca Jogado (0h): 13 título(s)
   - Casual Intermediário (10h - 100h): 8 título(s)
   - Engajado Avançado (100h - 500h): 6 título(s)

## 8. Insights Gerados

In [28]:
print("=" * 65)
print(" INSIGHTS FINAIS DO PROJETO")
print("=" * 65)

genero_top = df["Genero"].value_counts().idxmax()
print(f"\n1) Qual gênero domina minha biblioteca?")
print(f"   -> {genero_top} ({df['Genero'].value_counts().max()} jogos).")

perfil_top = df["Categoria_Consumo"].value_counts().idxmax()
print(f"\n2) Qual meu perfil de consumo predominante?")
print(f"   -> {perfil_top} ({df['Categoria_Consumo'].value_counts().max()} jogos nessa faixa).")

inativos = (df["Status_Ciclo_3_Anos"] == "Inativo (Mais de 3 Anos)").sum()
print(f"\n3) Quantos jogos eu \'esqueci\' (sem login há mais de 3 anos)?")
print(f"   -> {inativos} jogos.")

custo_beneficio = df[df["Custo_Por_Hora"].notna()].nsmallest(1, "Custo_Por_Hora")
if not custo_beneficio.empty:
    linha = custo_beneficio.iloc[0]
    print(f"\n4) Qual jogo teve o melhor custo-benefício (menor custo por hora, entre os com preço conhecido)?")
    print(f"   -> {linha['Nome_Jogo']}: US$ {linha['Custo_Por_Hora']}/hora.")

media_aprovacao = df["Percentual_Positivo"].mean()
print(f"\n5) Qual a aprovação média (crítica) dos jogos da biblioteca (entre os com avaliações disponíveis)?")
print(f"   -> {media_aprovacao:.1f}% de avaliações positivas, em média.")

print("\n" + "=" * 65)


 INSIGHTS FINAIS DO PROJETO

1) Qual gênero domina minha biblioteca?
   -> Action (8 jogos).

2) Qual meu perfil de consumo predominante?
   -> Experimental / Testado (< 10h) (23 jogos nessa faixa).

3) Quantos jogos eu 'esqueci' (sem login há mais de 3 anos)?
   -> 20 jogos.

4) Qual jogo teve o melhor custo-benefício (menor custo por hora, entre os com preço conhecido)?
   -> Counter-Strike 2: US$ 0.0/hora.

5) Qual a aprovação média (crítica) dos jogos da biblioteca (entre os com avaliações disponíveis)?
   -> 78.9% de avaliações positivas, em média.

